# Capstone 1: UK Road Safety

The training wheels are off.

Every notebook up to this point has taught you something and then asked
you to practise it in a controlled setting. This one is different. You
have a brief, you have data, and you have all the skills you have built
across the programme. Now put them together.

## The brief

The Department for Transport publishes road accident statistics for
Great Britain every year. The dataset we have is a sample of the
STATS19 returns for 2023 -- roughly 3,000 reported accidents with
details about location, conditions, severity, and timing.

Your job:

1. **Profile** the data -- understand its shape, spot the problems
2. **Clean** it -- fix the issues you find
3. **Load** it into a SQLite database with a sensible schema
4. **Analyse** it with SQL to answer specific questions about road safety
5. **Reflect** on what a production pipeline would need

This is a guided capstone. We will tell you what to do at each stage,
but you write the code. The guidance gets lighter as you go.

## Setup

In [ ]:
%pip install -q matplotlib jupysql duckdb-engine

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import sqlite3

---

## Step 1: Profile the data

Before you do anything clever, you need to understand what you are
working with. Load the CSV, then answer these questions:

- How many rows and columns?
- What are the data types? Do they match what you would expect?
- Which columns have missing values, and how many?
- What do the numeric distributions look like?
- What are the unique categories in the categorical columns?

You know how to do all of this. `.info()`, `.describe()`,
`.value_counts()`, `.isna().sum()` -- these are your profiling toolkit.

### Your turn

Load `road_accidents_sample.csv` from the `../data/` directory and
print the shape.

In [ ]:
# Load the data
df = pd.read_csv("../data/road_accidents_sample.csv")
print(f"{len(df)} rows, {len(df.columns)} columns")
df.head()

### Your turn

Run `.info()` on the DataFrame. Pay attention to the data types and
non-null counts. Are there columns that should be numeric but are
showing as `object`? That is a clue.

In [ ]:
# Your code here


### Your turn

Run `.describe()` for the numeric columns. Then calculate the null
percentage for every column and sort descending. Which columns have
the most missing data?

In [ ]:
# Your code here


### Your turn

Use `.value_counts()` to examine the categorical columns. Check at
least: `severity`, `road_type`, `weather_conditions`, and
`urban_rural`. Look for:

- Inconsistent casing (e.g. "Slight" vs "slight" vs "SLIGHT")
- Unexpected categories
- Suspicious distributions

In [ ]:
# Your code here


### Your turn

Create a histogram of `number_of_casualties` and a bar chart of
accident `severity` counts. These give you a visual sense of the
data before you start cleaning.

In [ ]:
# Your code here


### Your turn

Examine the `date` column. Most values should be in `YYYY-MM-DD`
format, but some might not be. Check a sample of values. Are there
any in `DD/MM/YYYY` format?

Also examine `speed_limit`. Are there any negative values or empty
strings?

In [ ]:
# Your code here


Good. By now you should have a mental list of problems. Write them
down in the cell below -- you will fix them in the next step.

### Your turn

**List the data quality issues you found:**

1. ...
2. ...
3. ...

(Edit this cell with your findings.)

---

## Step 2: Clean the data

Now fix the problems you found. Here is a checklist of things to
address. You may have spotted others.

- **Severity casing:** Standardise to title case ("Slight",
  "Serious", "Fatal")
- **Road type casing:** Same treatment
- **Date format:** Convert all dates to a consistent `YYYY-MM-DD`
  format. Some rows use `DD/MM/YYYY`.
- **Speed limit:** Convert to numeric. Handle empty strings and
  negative values (replace negatives with their absolute value --
  likely a data entry sign error).
- **Missing values:** Replace empty strings with `NaN` where
  appropriate. Decide what to do with nulls in each column.
- **Latitude/longitude:** Some rows have empty coordinates. Drop
  them or fill with `NaN`.

Work through these one at a time. After each fix, verify it worked.

### Your turn

Replace empty strings with `NaN` across the whole DataFrame.
Then standardise the `severity` and `road_type` columns to title case.

In [ ]:
# Replace empty strings with NaN
df = df.replace("", pd.NA)

# Your code to standardise severity and road_type


### Your turn

Fix the `date` column. You need to handle two formats:
`YYYY-MM-DD` and `DD/MM/YYYY`. One approach is to use
`pd.to_datetime` with `format='mixed'` or `dayfirst=True`, or
you could write a function that detects the format and converts.

After converting, verify by checking the min and max dates.

In [ ]:
# Your code here


### Your turn

Fix the `speed_limit` column:

1. Convert to numeric (use `pd.to_numeric` with `errors='coerce'`)
2. Replace negative values with their absolute value
3. Verify the result with `.value_counts()`

In [ ]:
# Your code here


### Your turn

Convert `latitude` and `longitude` to numeric, coercing errors.
Then run your profiling checks again on the cleaned DataFrame to
confirm the issues are resolved.

In [ ]:
# Your code here


---

## Step 3: Load into SQLite

Now we move from pandas to SQL. Create a SQLite database and load
the cleaned data into it.

Think about the schema. You could load everything into a single
`accidents` table -- that is fine for this exercise. If you want
to stretch yourself, you could normalise it: a `dim_location` table
for local authority and coordinates, a `dim_conditions` table for
weather, light, and road surface. But a single flat table is
perfectly acceptable here.

Use `pandas.DataFrame.to_sql()` to load the data, or write
CREATE TABLE and INSERT statements manually if you prefer more
control.

### Your turn

Create a SQLite connection (in-memory or file-based, your choice)
and load the cleaned DataFrame. Then verify by running a COUNT
query.

In [ ]:
# Your code here


### Setting up jupysql

Let's connect jupysql so we can write SQL directly in notebook cells.
If you loaded into a file-based database, adjust the path below.

In [ ]:
%load_ext sql
%sql sqlite:///road_safety.db

If you used an in-memory database, you will need to either:
- Re-load into a file-based DB (e.g. `road_safety.db`), or
- Use pandas and `conn.execute()` for your queries instead of jupysql

Either approach works. Pick whichever you are more comfortable with.

---

## Step 4: Analyse with SQL

Now the interesting part. Answer the following questions using SQL
queries. For each question, write the query and then add a brief
interpretation of the results in a markdown cell.

### Task 1: Which road types have the most accidents?

Group by `road_type`, count accidents, and order descending.
Also calculate the percentage of total accidents for each road type.

In [ ]:
# Your SQL query here


### Task 2: What time of day is most dangerous?

Extract the hour from the `time` column and group by it. Count
accidents per hour. A bar chart would be a good way to present this.

**Hint:** In SQLite, you can use `SUBSTR(time, 1, 2)` to extract
the hour, or `CAST(SUBSTR(time, 1, 2) AS INTEGER)` to get it as
a number.

In [ ]:
# Your SQL query here


### Task 3: How does weather affect accident severity?

This is a cross-tabulation question. For each weather condition,
show the count and percentage of each severity level.

You could do this with a GROUP BY on both columns, or use CASE
expressions to pivot. The question you are really asking: is the
proportion of fatal or serious accidents higher in bad weather?

In [ ]:
# Your SQL query here


### Task 4: Are accidents more common on certain days of the week?

Group by `day_of_week` and count. Order the days logically
(Monday through Sunday), not alphabetically.

**Bonus:** Also break this down by severity. Are fatal accidents
distributed differently across the week than slight accidents?

In [ ]:
# Your SQL query here


### Task 5: Urban vs rural accident profiles

Compare urban and rural accidents. Which has more? Which has a
higher proportion of serious/fatal accidents? What about the
average number of vehicles involved?

This is where the analysis gets genuinely interesting. Rural
roads tend to have higher speed limits and fewer witnesses.

In [ ]:
# Your SQL query here


### Task 6 (stretch): Speed limit and severity

Is there a relationship between speed limit and accident severity?
Group by speed limit and calculate the percentage of serious/fatal
accidents. Does higher speed limit correlate with worse outcomes?

This is the kind of question a transport policy analyst would
actually ask.

In [ ]:
# Your SQL query here


---

## Step 5: Reflect

You have built a complete pipeline: raw CSV to profiled, cleaned
data in a database, with analytical queries producing insights.
That is a real piece of data engineering work.

But it is also a simplified version of what happens in production.
Take a few minutes to think about these questions and write your
answers below.

### Your turn

**1. What would you do differently if this ran every day?**

Think about: idempotency, incremental loads, error handling,
logging, alerting on data quality thresholds.

*Your answer:*

...

**2. What validation rules would you add?**

Think about: Pydantic models, range checks on latitude/longitude,
allowed values for severity and road type, date range constraints.

*Your answer:*

...

**3. How would you test this pipeline?**

Think about: unit tests for cleaning functions, integration tests
with known input/output pairs, data quality checks that run
after every load.

*Your answer:*

...

**4. What would the schema look like at scale?**

If this were millions of rows across many years, would you still
use a single flat table? Or would you normalise, use a star schema,
or adopt a columnar format like Parquet?

*Your answer:*

...

---

## Summary

You have just completed your first end-to-end data pipeline without
someone writing most of the code for you. That is a significant
step.

The pattern you followed -- profile, clean, load, analyse -- is the
same pattern used in production systems at organisations from the
DfT to the NHS to multinational banks. The tools are sometimes
different (Spark instead of pandas, Snowflake instead of SQLite),
but the thinking is identical.

In the next capstone, the guidance drops further. You will deal
with multiple data sources in different formats, design a star
schema, and build a proper ETL pipeline with validation. The brief
gets shorter. The decisions are yours.